In [ ]:
%run NB00_setup

In [ ]:
%run NB00b_gold_config

# NB02 – Facts
Loads: `gold_fact_telemetry` (one row per reading), `gold_fact_trip` (one row per trip) and `gold_fact_anomaly_episode` (one row per episode).
Every load uses MERGE, so running it twice does not create duplicate rows.

In [ ]:
T_ALL = time.time()

def upsert(df, table, on, partition_by=None, update_cond=None):
    """First run: create the table. Later runs: MERGE (update when something changed, insert when new)."""
    if not table_exists(table):
        w = df.write.format("delta")
        if partition_by:
            w = w.partitionBy(partition_by)
        w.saveAsTable(table)
        return "created"
    m = DeltaTable.forName(spark, table).alias("t").merge(df.alias("s"), on)
    m = m.whenMatchedUpdateAll(condition=update_cond)
    m.whenNotMatchedInsertAll().execute()
    return "merged"

# ---------- 0.a) anomaly types: 'none' + any type found in silver that the dimension does not know yet ----------
none_row = spark.createDataFrame(
    [(0, "none", "No anomaly", "reading is not part of any anomaly", "gold load")],
    "anomaly_type_key long, anomaly_type string, anomaly_name string, definition string, source string")
merge_simple(none_row, "gold_dim_anomaly_type", "anomaly_type")

found = (spark.table(SRC_TELEMETRY).select("anomaly_type")
         .unionByName(spark.table(SRC_EPISODES).select("anomaly_type"))
         .where("anomaly_type IS NOT NULL").distinct())
known = spark.table("gold_dim_anomaly_type").select("anomaly_type")
newt = found.join(known, "anomaly_type", "left_anti")
n_new = newt.count()
if n_new:
    base = max_key("gold_dim_anomaly_type", "anomaly_type_key")
    add = (newt.withColumn("anomaly_type_key",
                           (F.row_number().over(Window.orderBy("anomaly_type")) + F.lit(base)).cast("long"))
           .withColumn("anomaly_name", F.initcap(F.regexp_replace("anomaly_type", "_", " ")))
           .withColumn("definition", F.lit("found in silver, not in the work plan"))
           .withColumn("source", F.lit("discovered in silver"))
           .select("anomaly_type_key", "anomaly_type", "anomaly_name", "definition", "source"))
    merge_simple(add, "gold_dim_anomaly_type", "anomaly_type")
print("new anomaly types added:", n_new)
spark.table("gold_dim_anomaly_type").orderBy("anomaly_type_key").show(truncate=False)

# ---------- 0.b) dim_date: extend the table if the data falls outside the existing range ----------
mm = spark.table(SRC_TELEMETRY).agg(F.min(F.to_date("event_time")).alias("mn"),
                                    F.max(F.to_date("event_time")).alias("mx")).first()
cur = spark.table("gold_dim_date").agg(F.min("full_date").alias("mn"), F.max("full_date").alias("mx")).first()
lo = min(str(mm["mn"]), str(cur["mn"]))
hi = max(str(mm["mx"]), str(cur["mx"]))
if lo < str(cur["mn"]) or hi > str(cur["mx"]):
    dd_ = spark.sql(f"SELECT explode(sequence(to_date('{lo}'), to_date('{hi}'), interval 1 day)) AS full_date")
    dim_date = dd_.select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"), "full_date",
        F.year("full_date").alias("year"), F.quarter("full_date").alias("quarter"),
        F.month("full_date").alias("month"), F.date_format("full_date", "MMMM").alias("month_name"),
        F.dayofmonth("full_date").alias("day"), F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"), F.weekofyear("full_date").alias("week_of_year"),
        F.dayofweek("full_date").isin(6, 7).alias("is_weekend"))
    merge_simple(dim_date, "gold_dim_date", "date_key")
    print("dim_date extended to", lo, "->", hi)
else:
    print("dim_date already covers", lo, "->", hi)

# ---------- 0.c) diagnostic: how many trip_ids exist in ref_trips? ----------
tr_s = spark.table(SRC_TELEMETRY).select("trip_id").where("trip_id IS NOT NULL").distinct()
n_tr = tr_s.count()
n_tr_match = tr_s.join(spark.table("ref_trips").select("trip_id"), "trip_id", "left_semi").count()
print(f"trip_ids in silver: {n_tr} | found in ref_trips: {n_tr_match}  "
      f"(the rest use the vehicle's main corridor)")

In [ ]:
# ======================= gold_fact_telemetry =======================
t0 = time.time()
cols = ["event_id", "trip_id", "vehicle_id", "event_time", "latitude", "longitude", "speed_kmh", "rpm",
        "engine_temp_c", "fuel_rate_lph", "odometer_km", "cargo_temp_c", "vehicle_type", "assigned_driver_id",
        "time_gap_sec", "distance_km", "calculated_speed_kmh", "is_idling", "is_cold_chain_violation",
        "is_gps_teleportation", "is_engine_overheating", "is_rpm_spike", "is_cold_chain_breach",
        "is_excessive_idling", "is_anomaly", "anomaly_type"]

src = dedupe_telemetry(spark.table(SRC_TELEMETRY).select(*cols))
src = add_corridor(src)
src = src.withColumn("at_name", F.coalesce(F.col("anomaly_type"), F.lit("none")))

dv  = F.broadcast(spark.table("gold_dim_vehicle").select("vehicle_id", "vehicle_key"))
dr  = F.broadcast(spark.table("gold_dim_route").select("corridor_id", "route_key"))
dat = F.broadcast(spark.table("gold_dim_anomaly_type")
                  .select(F.col("anomaly_type").alias("at_name"), "anomaly_type_key"))
ddr = F.broadcast(spark.table("gold_dim_driver")
                  .select(F.col("driver_id").alias("d_driver_id"), "driver_key", "effective_from", "effective_to"))

j = (src.join(dv, "vehicle_id", "left")
        .join(dr, "corridor_id", "left")
        .join(dat, "at_name", "left")
        # SCD2 point-in-time: the driver version that was valid at the reading time
        .join(ddr, (F.col("assigned_driver_id") == F.col("d_driver_id"))
                   & (F.col("event_time") >= F.col("effective_from"))
                   & (F.col("event_time") < F.col("effective_to")), "left"))

fact = j.select(
    "event_id", "trip_id", "vehicle_key", "driver_key", "route_key", "anomaly_type_key",
    F.date_format("event_time", "yyyyMMdd").cast("int").alias("date_key"),
    (F.hour("event_time") * 100 + F.minute("event_time")).cast("int").alias("time_key"),
    "event_time", "latitude", "longitude", "speed_kmh", "rpm", "engine_temp_c", "fuel_rate_lph",
    "odometer_km", "cargo_temp_c",
    F.when(valid_iv, F.col("time_gap_sec")).otherwise(F.lit(0)).cast("long").alias("interval_sec"),
    distance_expr.alias("distance_km"),
    fuel_expr.alias("fuel_used_l"),
    idle_min_expr.alias("idle_minutes"),
    cold_total_expr.alias("cold_reading_cnt"),
    cold_ok_expr.alias("cold_in_range_cnt"),
    speeding_expr.alias("speeding_cnt"),
    flag("is_idling").cast("int").alias("idling_flag"),
    flag("is_cold_chain_violation").cast("int").alias("cold_violation_flag"),
    flag("is_gps_teleportation").cast("int").alias("gps_teleport_flag"),
    flag("is_engine_overheating").cast("int").alias("overheating_flag"),
    flag("is_rpm_spike").cast("int").alias("rpm_spike_flag"),
    flag("is_cold_chain_breach").cast("int").alias("cold_breach_flag"),
    flag("is_excessive_idling").cast("int").alias("excessive_idling_flag"),
    flag("is_anomaly").cast("int").alias("anomaly_flag"))

res = upsert(fact, "gold_fact_telemetry",
             on="t.event_id = s.event_id AND t.date_key = s.date_key",
             partition_by="date_key",
             update_cond="NOT (t.driver_key <=> s.driver_key) OR NOT (t.route_key <=> s.route_key) "
                         "OR NOT (t.anomaly_type_key <=> s.anomaly_type_key) OR NOT (t.vehicle_key <=> s.vehicle_key) "
                         "OR NOT (t.fuel_used_l <=> s.fuel_used_l) OR NOT (t.distance_km <=> s.distance_km) "
                         "OR NOT (t.idle_minutes <=> s.idle_minutes)")
n_fact = spark.table("gold_fact_telemetry").count()
print("gold_fact_telemetry:", res, "| rows =", n_fact)
log_run("NB02 fact_telemetry", t0, n_fact)

In [ ]:
# ======================= gold_fact_trip =======================
t0 = time.time()
ft = spark.table("gold_fact_telemetry").where("trip_id IS NOT NULL")
trip = (ft.groupBy("trip_id").agg(
            F.min(F.struct("event_time", "vehicle_key", "driver_key", "route_key")).alias("first_row"),
            F.min("event_time").alias("trip_start"),
            F.max("event_time").alias("trip_end"),
            F.count("*").alias("reading_count"),
            F.sum("distance_km").alias("distance_km"),
            F.sum("fuel_used_l").alias("fuel_used_l"),
            F.sum("idle_minutes").alias("idle_minutes"),
            F.sum("cold_reading_cnt").alias("refrigerated_readings_total"),
            F.sum("cold_in_range_cnt").alias("refrigerated_readings_in_range"),
            F.sum("speeding_cnt").alias("speeding_readings"),
            F.sum("anomaly_flag").alias("anomaly_readings"))
        .select("trip_id",
                F.col("first_row.vehicle_key").alias("vehicle_key"),
                F.col("first_row.driver_key").alias("driver_key"),
                F.col("first_row.route_key").alias("route_key"),
                F.date_format("trip_start", "yyyyMMdd").cast("int").alias("start_date_key"),
                "trip_start", "trip_end",
                ((F.unix_timestamp("trip_end") - F.unix_timestamp("trip_start")) / 60.0).alias("duration_minutes"),
                "reading_count", "distance_km", "fuel_used_l", "idle_minutes",
                "refrigerated_readings_total", "refrigerated_readings_in_range",
                "speeding_readings", "anomaly_readings"))
res = upsert(trip, "gold_fact_trip", on="t.trip_id = s.trip_id")
n_trip = spark.table("gold_fact_trip").count()
print("gold_fact_trip:", res, "| rows =", n_trip)
log_run("NB02 fact_trip", t0, n_trip)

In [ ]:
# ======================= gold_fact_anomaly_episode =======================
t0 = time.time()
ep = (spark.table(SRC_EPISODES)
      .withColumn("episode_nk", F.concat_ws("|", "vehicle_id", "anomaly_type", F.col("episode_id").cast("string")))
      .dropDuplicates(["episode_nk"]))
ep = add_corridor(ep, has_trip=False)

dvv = F.broadcast(spark.table("gold_dim_vehicle").select("vehicle_id", "vehicle_key", "assigned_driver_id"))
dat2 = F.broadcast(spark.table("gold_dim_anomaly_type").select("anomaly_type", "anomaly_type_key"))

ep = (ep.join(dvv, "vehicle_id", "left")
        .join(dr, "corridor_id", "left")
        .join(dat2, "anomaly_type", "left")
        .join(ddr, (F.col("assigned_driver_id") == F.col("d_driver_id"))
                   & (F.col("start_time") >= F.col("effective_from"))
                   & (F.col("start_time") < F.col("effective_to")), "left"))

fact_ep = ep.select(
    "episode_nk", "episode_id", "vehicle_key", "driver_key", "route_key", "anomaly_type_key",
    F.date_format("start_time", "yyyyMMdd").cast("int").alias("start_date_key"),
    (F.hour("start_time") * 100 + F.minute("start_time")).cast("int").alias("start_time_key"),
    "start_time", "end_time", "duration_seconds",
    (F.col("duration_seconds") / 60.0).alias("duration_minutes"),
    "reading_count", "max_calculated_speed")

res = upsert(fact_ep, "gold_fact_anomaly_episode", on="t.episode_nk = s.episode_nk")
n_ep = spark.table("gold_fact_anomaly_episode").count()
print("gold_fact_anomaly_episode:", res, "| rows =", n_ep)
log_run("NB02 fact_anomaly_episode", t0, n_ep)

In [ ]:
# ======================= summary =======================
for t in ["gold_fact_telemetry", "gold_fact_trip", "gold_fact_anomaly_episode"]:
    print(f"{t:28s} rows = {spark.table(t).count()}")
print("silver_telemetry rows           =", spark.table("silver_telemetry").count())
print("silver_anomaly_episodes rows    =", spark.table(SRC_EPISODES).count())

print("\nBefore/after the driver change: readings of the same driver before and after 2026-11-03 12:00 point to different driver_key values")
chg = (spark.table("gold_dim_driver").groupBy("driver_id").count().where("count > 1").select("driver_id")
       .join(spark.table("gold_dim_vehicle").select(F.col("assigned_driver_id").alias("driver_id")).distinct(),
             "driver_id", "left_semi").limit(1))
if chg.count():
    did = chg.first()["driver_id"]
    (spark.table("gold_fact_telemetry").alias("f")
        .join(spark.table("gold_dim_driver").alias("d"), "driver_key")
        .where(F.col("d.driver_id") == did)
        .groupBy("d.driver_id", "d.driver_key", "d.home_depot_id", "d.risk_level",
                 "d.effective_from", "d.effective_to")
        .agg(F.count("*").alias("readings"), F.min("f.event_time").alias("first_reading"),
             F.max("f.event_time").alias("last_reading"))
        .orderBy("d.effective_from").show(truncate=False))
log_run("NB02 TOTAL", T_ALL)

show_history("gold_fact_telemetry", 5)